In [0]:

-- Alert 1: share of quarantined ticks in the last hour
-- Condition: quarantine_pct  >  20
WITH q AS (
  SELECT COUNT(*) AS c
  FROM dbr_dev_ua5816bd.team_crypto_silver.ticks_quarantine
  WHERE ingestion_timestamp >= current_timestamp() - INTERVAL 1 HOUR
),
s AS (
  SELECT COUNT(*) AS c
  FROM dbr_dev_ua5816bd.team_crypto_silver.ticks_silver
  WHERE ingestion_timestamp >= current_timestamp() - INTERVAL 1 HOUR
)
SELECT
  q.c AS quarantined_last_hour,
  s.c AS valid_last_hour,
  ROUND(100.0 * q.c / NULLIF(q.c + s.c, 0), 2) AS quarantine_pct
FROM q CROSS JOIN s;

-- Alert 2: bronze freshness
-- Condition: minutes_since_last_row  >  60

SELECT
  COALESCE(
    TIMESTAMPDIFF(MINUTE, MAX(ingestion_timestamp), current_timestamp()),
    999999
  ) AS minutes_since_last_row
FROM dbr_dev_ua5816bd.team_crypto_bronze.ticks_bronze;